## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [1]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.discriminant_analysis import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB



### Configuración de constantes, rutas y variables 

En esta sección definimos constantes, rutas de archivos y atributos del dataset

In [2]:
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Cargamos los datos
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender', 'Surname']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId']
SURNAME_COL = 'Surname'
RANDOM_STATE = 100            # Semilla para reproducibilidad

### 1. Preparación de Datos

Separamos variables independientes y dependientes en X_train e y_train por convención.

In [3]:
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

## Clase para añadir features 

Creamos una clase para crear features sin fuga de datos. Esta clase aprende estadísticos (medianas y frecuencias) durante el fold de entrenamiento (`fit()`) y crea columnas nuevas usando esos estadístico, sin mirar la variable objetivo y sin fuga de datos tranformación(`tranform()`). No imputa ni escala de forma definitiva.

Activamos por familias.



In [4]:
# ---------- Feature Engineering ----------
from sklearn.base import BaseEstimator, TransformerMixin
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Genera features nuevas de forma segura (sin fuga):
    aprende mediana/frecuencias en fit() y las usa en transform().
    Activación por familias para ir de menos a más.
    """
    def __init__(
        self,
        # cada flag activa un grupo de features nuevas para probar mejoras
        # de forma incremental
        add_missing_count=True,     # cuenta de missings
        add_balance=True,           # añade features de Balance/Salary/ratios
        add_products=True,           # añade features de NumOfProducts
        add_age=True,                # añade features de Age/grupos de edad
        add_age_bins=True,           # añade variable categórica por tramos de edad
        add_interactions=True,       # añade features de interacciones simples
        add_surname_features=False   # añade features basadas en Surname
    ):
        self.add_missing_count = add_missing_count
        self.add_balance = add_balance
        self.add_products = add_products
        self.add_age = add_age
        self.add_age_bins = add_age_bins
        self.add_interactions = add_interactions
        self.add_surname_features = add_surname_features
   
    def fit(self, X, y=None):
        X = X.copy()
        # Calculamos medianas de columnas clave
        # lo hacemos en fit porque en validación cruzada, cada fold tiene un “train interno”
        # Si calculamos las medianas con todo el dataset,estaríamos usando info del fold de validación 
        # (fuga de datos) hacia el train
        # La hacemos en fit() para garantizar que cada fold de CV aprende sus propias medianas
        # Usamos medianas para evitar outliers y crear features sin NaN
        self.balance_median_ = X["Balance"].median(skipna=True)
        self.salary_median_ = X["EstimatedSalary"].median(skipna=True)
        self.creditscore_median_ = X["CreditScore"].median(skipna=True)
        self.age_median_ = X["Age"].median(skipna=True)
        # para la variable NumOfProducts (discreta) usamos moda (valor más frecuente)
        self.numprod_mode_ = X["NumOfProducts"].mode(dropna=True).iloc[0]

        # Calculamos frecuencia de apellidos para hacer frequency encoding (opcional)
        # tiene riesgo si en test hay apellidos no vistos en train
        # lo hacemos en fit() para evitar fuga de datos
        # Evitaría usar one-hot encoding de Surname por 
        # el alto cardinalidad (enorme número de categorías si cada cliente tiene un apellido distinto)
        # a veces mete ruido si hay muchos apellidos únicos
        if self.add_surname_features and "Surname" in X.columns:
            s = X["Surname"].astype("object")
            self.surname_freq_ = s.value_counts(dropna=True)
        else:
            self.surname_freq_ = None

        return self

    def transform(self, X):
        # Copia para retornarlo con las nuevas features añadidas
        X = X.copy()
        
        # --- Missing count por fila ---
        # add_indicator ya añade columnas binarias (crea una feature) por cada columna con missings
        # pero aquí añadimos una feature con el conteo total de missings por fila
        # La falta de datos indica el perfil del cliente,
        # un cliente con muchos datos faltantes puede ser menos comprometido, menos activo, menos interesado, etc.
        if self.add_missing_count:
            # columnas a considerar para el conteo de missings
            miss_cols = ["CreditScore","Balance","NumOfProducts","EstimatedSalary","HasCrCard"]
            # solo las que existan
            miss_cols = [c for c in miss_cols if c in X.columns]
            X["missing_count"] = X[miss_cols].isna().sum(axis=1)

        # --- Balance / Salary / ratios ---
        # features basadas en Balance y EstimatedSalary
        # Creamos "señales" de tipo "tengo saldo o no", "saldo cero", etc.
        # Creamos features logarítmicas para reducir el impacto de outliers y colas largas
        # los logaritmos ayudan a modelos lineales a capturar relaciones no lineales
        # La relación entre Balance y Salary puede indicar el nivel de ahorro o gasto del cliente
        # Creamos ratios entre Balance y Salary y su logaritmo para capturar la relación entre ambos
        # Si un cliente tiene un balance alto en comparación con su salario, puede indicar una mayor estabilidad financiera
        # o capacidad de ahorro, lo cual puede influir en su probabilidad de abandono
        if self.add_balance:
            balance_raw = bal_raw = pd.to_numeric(X["Balance"], errors="coerce")   # strings -> NaN
            balance = balance_raw.fillna(self.balance_median_)
            balance_0 = bal_raw.fillna(0)
            salary = X["EstimatedSalary"].fillna(self.salary_median_)
            # features binarias
            X["HasBalance"] = (balance_0 > 0).astype(int)     # 1 si Balance > 0 (NaN -> 0)
            X["Balance_is_zero"] = (balance_0 == 0).astype(int) # 1 si Balance == 0 (NaN -> 0).
            # features logarítmicas
            X["LogBalance"] = np.log1p(np.clip(balance, 0, None))
            X["LogSalary"] = np.log1p(np.clip(salary, 0, None))
            # ratios Balance/Salary
            X["BalToSal"] = balance / (salary + 1.0) # evitar división por cero
            X["LogBalToSal"] = np.log1p(np.clip(X["BalToSal"], 0, None)) # log(1 + ratio)
        
        # --- Número de productos (saltos típicos) ---
        # features basadas en NumOfProducts
        # NumOfProducts es una variable NO lineal (1->2->3) y va por saltos
        # así que creamos features binarias para cada salto típico
        # 1 producto, 2 productos, 3 o más productos
        if self.add_products:
            num_products = pd.to_numeric(X["NumOfProducts"], errors="coerce")
            num_products = num_products.fillna(self.numprod_mode_)
            X["Products_eq1"] = (num_products == 1).astype(int)
            X["Products_eq2"] = (num_products == 2).astype(int)
            X["Products_ge3"] = (num_products >= 3).astype(int)

        # --- Edad ---
        # features basadas en Age
        # Age tiene una relación no lineal con la retención de clientes
        # así que creamos Age al cuadrado para capturar esa no linealidad
        if self.add_age:
            X["Age2"] = X["Age"] ** 2
        
        #--- Age por tramos (bins) (opcional) ---
        # crea una variable categórica por tramos de edad
        # La edad puede influir en el comportamiento del cliente y su probabilidad de abandono
        # Creamos grupos de edad para capturar patrones específicos en diferentes rangos de edad
        # útil para modelos no lineales o árboles
        # puede meter ruido
        if self.add_age_bins:
            X["AgeBin"] = pd.cut(
                X["Age"],
                bins=[17, 25, 35, 45, 55, 100],
                labels=["18-25", "26-35", "36-45", "46-55", "56+"],
                include_lowest=True
            ).astype("object")
        
        # --- Interacciones simples ---
        # crea features de interacciones simples entre variables clave
        # interacciones típicas: 
        # Usuario inactivo si IsActiveMember == 0
        # Usuario inactivo y que tenga más de 3 productos (puede indicar un cliente 
        # con muchos productos pero poco comprometido)
        # País (geografía) y género para capturar si el efecto de género varía por país o
        # como indica la Female tiene más abandono en general
        if self.add_interactions:
            X["Inactive"] = (X["IsActiveMember"] == 0).astype(int)

            # interacción muy común: inactivo + 3+ productos
            # Los clientes con
            if "Products_ge3" in X.columns:
                X["Inactive_ge3prod"] = ((X["Inactive"] == 1) & (X["Products_ge3"] == 1)).astype(int)
            

            # interacción Geografía/género (Germany/Female)
            # crea variables binarias para cada combinación
            # one-hot encoding manual
            # Los alemanes son lo que más abandonan
            X["Germany"] = (X["Geography"] == "Germany").fillna(False).astype(int) # 1 si Germany 0 si no
            #X["France"] = (X["Geography"] == "France").fillna(False).astype(int)
            #X["Spain"] = (X["Geography"] == "Spain").fillna(False).astype(int)
            # Género female?
            X["Female"] = (X["Gender"] == "Female").fillna(False).astype(int)
            # interacciones female/pais
            X["Female_Germany"] = ((X["Female"] == 1) & (X["Germany"] == 1)).astype(int)
            #X["Female_France"] = ((X["Female"] == 1) & (X["France"] == 1)).astype(int)
            #X["Female_Spain"] = ((X["Female"] == 1) & (X["Spain"] == 1)).astype(int)
        
         # --- Surname ---
        # features basadas en Surname (opcional y más riesgoso)
        # añade features sobre Surname: missing, longitud y frecuencia
        # Frecuncia del apellido para 
        # tiene riesgo si en test hay apellidos no vistos en train
        if self.add_surname_features and "Surname" in X.columns:
            surname = X["Surname"].astype("object") # 
            X["Surname_missing"] = surname.isna().astype(int)   # 1 si Surname es NaN
            X["Surname_len"] = surname.fillna("").astype(str).str.len().astype(int) # longitud del apellido 0 si es NaN len(surname)
            # frecuencia del apellido (frequency encoding)
            if self.surname_freq_ is not None:
                X["Surname_freq"] = surname.map(self.surname_freq_).fillna(0).astype(float)
            else:
                X["Surname_freq"] = 0.0
        
        return X  # retornamos el DataFrame con las nuevas features añadidas

#### Construir el preprocesador

In [5]:
# ---------- Preprocessor builder ----------
from sklearn.preprocessing import PowerTransformer


def make_preprocessor(use_surname: bool = False):
    # Columnas base
    numerical_continuous_cols = ["CreditScore", "Age", "Balance", "EstimatedSalary"]
    numerical_discrete_cols = ["Tenure", "NumOfProducts"]
    numerical_binary_cols = ["HasCrCard", "IsActiveMember"]

    # --- Columnas features por grupo ---
    continuous_fe = []  # continuas creadas 
    discrete_fe = []  # discretas creadas
    binary_fe  = []  # binarias creadas

    # features que pueden existir según flags de FeatureEngineer
    discrete_fe += ["missing_count"]
    continuous_fe += ["LogBalance", "LogSalary", "BalToSal", "LogBalToSal", "Age2"]
    binary_fe  += ["HasBalance", "Balance_is_zero", 
                   "Products_eq1", "Products_eq2", "Products_ge3",
                   "Inactive", "Inactive_ge3prod", 
                   "Germany","Female", #"France","Spain"
                   "Female_Germany"]#,"Female_France","Female_Spain"]
    
    # Surname features (opcional)
     # tiene riesgo si en test hay apellidos no vistos en train
     # añade features sobre Surname: missing, longitud y frecuencia
     # Evitaría usar one-hot encoding de Surname por 
     # el alto cardinalidad (enorme número de categorías si cada cliente tiene un apellido distinto)
     # a veces mete ruido si hay muchos apellidos únicos
     # lo activamos solo si use_surname == True
    if use_surname:
        discrete_fe += ["Surname_len"]
        binary_fe  += ["Surname_missing"]
        continuous_fe += ["Surname_freq"]

    # --- Pipelines numéricos ---
    continuous_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("power", PowerTransformer(method="yeo-johnson")),
        ("scaler", RobustScaler()),
    ])

    # Feature Eng continuas: mejor NO aplicar power otra vez 
    continuous_fe_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", RobustScaler()), # RobustScaler para features con outliers
    ])

    discrete_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ])

    binary_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ])

    # --- Categóricas ---
    cat_cols = ["Geography", "Gender", "AgeBin"]
    
    # ONE HOT ENCODER para convertir categóricas en numericas
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
    )
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ])

    transformers = [
        ("cont", continuous_pipe, numerical_continuous_cols),
        ("cont_eng", continuous_fe_pipe, continuous_fe),
        ("disc", discrete_pipe, numerical_discrete_cols + discrete_fe),
        ("bin", binary_pipe, numerical_binary_cols + binary_fe),
        ("cat", categorical_pipe, cat_cols),
    ]

    preprocesor = ColumnTransformer(
        transformers=transformers,  # transformers 
        remainder="drop",           # elimina columnas no especificadas
        verbose_feature_names_out=True, # nombres detallados de columnas
        sparse_threshold=0.0 
    )

    return preprocesor

#### Construir pipelines

In [6]:
# Creación y evaluación del pipeline
# Construcción del pipeline con feature engineer, preprocesador y modelo

def make_pipeline(feature_engineer:FeatureEngineer, preprocessor: ColumnTransformer, model=None):
    """Construye un Pipeline con el preprocesador y el modelo indicado.
    Args:
        feature_engineer (FeatureEngineer): Objeto FeatureEngineer para crear nuevas features
        preprocessor (ColumnTransformer): Preprocesador ColumnTransformer
        model (_type_, optional): Modelo de clasificación. Defaults to None.
    Returns:
        Pipeline: Pipeline con preprocesador y modelo, si no se indica modelo
        se usa LinearDiscriminantAnalysis por defecto.
    """
    if model is None:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        #model = LogisticRegression(max_iter=10000,solver="saga", random_state=RANDOM_STATE,class_weight='balanced',n_jobs=-1)
    if feature_engineer is None:
        return Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", model),
        ])
    else:
        return Pipeline([
        ("feature_engineer", feature_engineer),
        ("preprocessor", preprocessor),
        ("classifier", model),
        ])




#### Evaluar pipelines

Esta función evalua el pipeline usando validación cruzada estratificada

In [7]:
def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, n_splits=5):
    """ Evalúa el pipeline usando Validación Cruzada estratificada
    Args:
        pipe (Pipeline): Pipeline a evaluar.
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        dict: Diccionario con las métricas promedio y desviación estándar.
    """
    # Configuramos la Validación Local Cruzada  n_splits splits (divisiones)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    # Definimos las métricas que queremos extraer
    # f1, roc_auc, precision, recall, accuracy son strings estándar de sklearn.
    # Kappa requiere make_scorer.
    scoring_metrics = {
        "f1": "f1",
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        'kappa': make_scorer(cohen_kappa_score),
        'precision': 'precision',
        'recall': 'recall',
    }
    cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    return {
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std":  cv_results["test_f1"].std(),
        "auc_mean": cv_results["test_roc_auc"].mean(),
        "auc_std":  cv_results["test_roc_auc"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std":  cv_results["test_accuracy"].std(),
        "kappa_mean": cv_results["test_kappa"].mean(),
        "kappa_std":  cv_results["test_kappa"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std":  cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std":  cv_results["test_recall"].std()
    }

------------------
## Evaluación de diferentes features engineer y modelos

Probamos las diferentes variables que vamos creando en la clase `FeatureEngineer`. 

In [8]:
# Configuración de experimentos
# modificamos Feature Engineer a probar 
# Feature Engineer 
feature_engineer = FeatureEngineer(
    add_missing_count=True,
    add_balance=True,
    add_products=True,
    add_age=True,
    add_age_bins=True,
    add_interactions=True,
    add_surname_features=True
)
# Hacemos el preprocesador con o sin features de Surname
use_surname_preprocesor = True



#### Búsqueda del mejor modelo

Evaluamos los modelos con el mejor preprocesador encontrado 

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.utils import compute_sample_weight

# Ahora hacemos la búsqueda de hiperparámetros con GridSearchCV
# Definir el modelo base
#model = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE,solver="newton-cg", class_weight="balanced")
model = RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced",n_jobs=-1,)
# Construimos el preprocesador fijo con la mejor versión
preprocesor = make_preprocessor(use_surname=True)
# Construimos el pipeline
pipe = make_pipeline(feature_engineer, preprocesor, model)
# Evaluamos el pipeline
cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
model_results = []
model_results.append({"modelo": "LogisticRegression", **cv_metrics})
display(pd.DataFrame(model_results).sort_values("f1_mean", ascending=False))

# Definir el "diccionario de rejilla" (parámetros a probar)

# --- Búsqueda de hiperparámetros ---
param_grid = {
    # Número de árboles en el bosque
    'classifier__n_estimators': [100, 200, 300],
    
    # Profundidad máxima del árbol (evita sobreajuste si se limita)
    'classifier__max_depth': [None, 10, 20, 30],
    
    # Mínimo de muestras requeridas para dividir un nodo
    'classifier__min_samples_split': [2, 5, 10],
    
    # Mínimo de muestras requeridas en un nodo hoja
    'classifier__min_samples_leaf': [1, 2, 4],
    
    # Selección de características para la mejor división (auto = sqrt)
    'classifier__max_features': ['sqrt', 'log2']
}
# Configuramos la búsqueda optimizando F1
search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=5,               # 5-Fold Cross Validation
    scoring='f1',       # <--- IMPORTANTE: Le decimos que maximice F1, no Accuracy
    n_jobs=-1,
    verbose=1
)

# Entrenamos con los datos completos de entrenamiento
search.fit(X_train, y_train)
# Resultados
print(f"Mejor F1 encontrado: {search.best_score_:.4f}")
print(f"Mejores parámetros: {search.best_params_}")

# objeto 'search' ahora se comporta como el mejor modelo encontrado
best_model_pipe = search.best_estimator_
#print("best_model:", best_model)

,modelo,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
0,LogisticRegression,0.533056,0.010247,0.841522,0.007362,0.850875,0.003549,0.452271,0.011906,0.736737,0.02046,0.417791,0.010155


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=5, classifier__n_estimators=100; total time=   1.4s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=5, classifier__n_estimators=100; total time=   1.9s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=300; total time=   3.8s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=300; total time=   3.8s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=100; total time=   1.8s
[CV] END classifier__max_d

/home/administrador/miniforge3/envs/tareaK_env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END classifier__max_depth=10, classifier__max_features=log2, classifier__min_samples_leaf=1, classifier__min_samples_split=5, classifier__n_estimators=200; total time=   2.9s
[CV] END classifier__max_depth=10, classifier__max_features=log2, classifier__min_samples_leaf=1, classifier__min_samples_split=10, classifier__n_estimators=100; total time=   1.6s
[CV] END classifier__max_depth=10, classifier__max_features=log2, classifier__min_samples_leaf=1, classifier__min_samples_split=5, classifier__n_estimators=200; total time=   2.8s
[CV] END classifier__max_depth=10, classifier__max_features=log2, classifier__min_samples_leaf=1, classifier__min_samples_split=5, classifier__n_estimators=200; total time=   2.8s
[CV] END classifier__max_depth=10, classifier__max_features=log2, classifier__min_samples_leaf=1, classifier__min_samples_split=10, classifier__n_estimators=100; total time=   1.6s
[CV] END classifier__max_depth=10, classifier__max_features=log2, classifier__min_samples_leaf=1, 

## Construcción del pipeline final para Kaggle

Una vez obtenido la mejor combinación parámetros para el modelo reentrenamos para finalmente obtener la predicción y generar el fichero para kaggle. 

In [10]:
# Construcción del pipeline final para Kaggle con el mejor modelo encontrado

# Configuramos y ejecutamos la Validación Cruzada local
cv_metrics = evaluate_pipeline(best_model_pipe, X_train, y_train, n_splits=5)
# Generación de Submission para Kaggle con el mejor modelo encontrado
# Re-entrenamos con TODOS los datos de train para la predicción final
best_model_pipe.fit(X_train, y_train) 
test_predictions = best_model_pipe.predict(test_df)

# Crear fichero de salida
submission_df = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

print("\n---- Mejores Resultados y Validación Cruzada local -----")
#print("Mejor modelo:", best_model)
#print("Mejor versión de preprocesador:", best_model)
print(f"Mean F1-Score:  {cv_metrics['f1_mean']:.4f} (+/- Std {cv_metrics['f1_std']:.4f})")
print(f"Mean Accuracy:  {cv_metrics['accuracy_mean']:.4f} (+/- Std {cv_metrics['accuracy_std']:.4f})")
print(f"Mean Kappa:     {cv_metrics['kappa_mean']:.4f}")
print(f"Mean Precision: {cv_metrics['precision_mean']:.4f}")
print(f"Mean Recall:    {cv_metrics['recall_mean']:.4f}")



Fichero '/kaggle/working/submission.csv' generado correctamente.

---- Mejores Resultados y Validación Cruzada local -----
Mean F1-Score:  0.5996 (+/- Std 0.0115)
Mean Accuracy:  0.8337 (+/- Std 0.0041)
Mean Kappa:     0.4949
Mean Precision: 0.5892
Mean Recall:    0.6117
